In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
#for dirname, _, filenames in os.walk('/kaggle/input'):
 #   for filename in filenames:
  #      print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

In [2]:
import torch
print(torch.cuda.is_available(), torch.cuda.get_device_name(0))

True Tesla T4


In [3]:
import os, torch
import torch.nn as nn
from torchvision import transforms, datasets
from torch.utils.data import DataLoader

ROOT = "/kaggle/input/datasets/valdivinosantiago/imagenettetvt320"
for d in os.listdir(ROOT):
    print(d)

val
test
train


In [4]:
class AlexNet(nn.Module):

    def __init__(self, num_classes=10, dropout = 0.5):
        super().__init__()
        
        self.conv1 = nn.Conv2d(in_channels=3, out_channels=64, kernel_size = 11, stride=4, padding=2)
        self.relu1 = nn.ReLU(inplace = True)
        self.pool1 = nn.MaxPool2d(kernel_size=3,stride=2)

        self.conv2 = nn.Conv2d(in_channels = 64, out_channels=192,kernel_size=5, stride=1, padding = 2)
        self.relu2 = nn. ReLU(inplace=True)
        self.pool2 = nn.MaxPool2d(kernel_size = 3, stride = 2)

        self.conv3 = nn.Conv2d(in_channels=192, out_channels = 384, kernel_size = 3, stride = 2)
        self.relu3 = nn.ReLU(inplace=True)

        self.conv4 = nn.Conv2d(in_channels = 384,out_channels = 256,kernel_size = 3,padding =1 )
        self.relu4 = nn.ReLU(inplace = True)

        self.conv5 = nn.Conv2d(in_channels=256, out_channels=256, kernel_size=3, padding=1)
        self.relu5 = nn.ReLU(inplace=True)
        self.pool5 = nn.MaxPool2d(kernel_size=3, stride=2)

        self.flatten = nn.Flatten()

        self.dropout1 = nn.Dropout(dropout)
        self.fc1 = nn.Linear(in_features = 4096,out_features = 4096)
        self.relu6 = nn.ReLU(inplace = True)

        self.dropout2 = nn.Dropout(dropout)
        self.fc2 = nn.Linear(in_features=4096, out_features=4096)
        self.relu7 = nn.ReLU(inplace=True)

        self.fc3 = nn.Linear(in_features=4096, out_features=num_classes)

    def forward(self,x):

        x = self.relu1(self.conv1(x))
        x = self.pool1(x)

        x = self.relu2(self.conv2(x))
        x= self.pool2(x)
        x = self.relu3(self.conv3(x))
        x = self.relu4(self.conv4(x))
        x = self.relu5(self.conv5(x))
        x = self.pool5(x)

        x = self.flatten(x)
        x = self.relu6(self.fc1(self.dropout1(x)))
        x = self.relu7(self.fc2(self.dropout2(x)))
        x = self.fc3(x)

        return x

In [5]:

train_tf = transforms.Compose([
    transforms.Resize(320),
    transforms.RandomCrop(320, padding=8),
    transforms.RandomHorizontalFlip(),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225]),
])
val_tf = transforms.Compose([
    transforms.Resize(320),
    transforms.CenterCrop(320),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225]),
])

train_ds = datasets.ImageFolder(f"{ROOT}/train", transform=train_tf)
val_ds   = datasets.ImageFolder(f"{ROOT}/val",   transform=val_tf)
test_ds  = datasets.ImageFolder(f"{ROOT}/test",  transform=val_tf)

train_dl = DataLoader(train_ds, batch_size=128, shuffle=True,  num_workers=2, pin_memory=True)
val_dl   = DataLoader(val_ds,   batch_size=128, shuffle=False, num_workers=2, pin_memory=True)
test_dl  = DataLoader(test_ds,  batch_size=128, shuffle=False, num_workers=2, pin_memory=True)

print("train:", len(train_ds), "val:", len(val_ds), "test:", len(test_ds))
print("classes:", train_ds.classes)

train: 9469 val: 1309 test: 2616
classes: ['cassetePlayer', 'chainShaw', 'church', 'englishSpringer', 'frenchHorn', 'garbageTruck', 'gasPump', 'golfBall', 'parachute', 'tench']


In [6]:
device = "cuda"
model = AlexNet(num_classes=10).to(device)

opt   = torch.optim.SGD(model.parameters(), lr=0.01, momentum=0.9, weight_decay=5e-4)
sched = torch.optim.lr_scheduler.StepLR(opt, step_size=15, gamma=0.1)
crit  = nn.CrossEntropyLoss()
scaler = torch.amp.GradScaler("cuda")

best_acc = 0.0
for epoch in range(1, 31):
    model.train()
    loss_sum = 0
    for x, y in train_dl:
        x, y = x.to(device, non_blocking=True), y.to(device, non_blocking=True)
        opt.zero_grad()
        with torch.amp.autocast("cuda"):
            loss = crit(model(x), y)
        scaler.scale(loss).backward()
        scaler.step(opt)
        scaler.update()
        loss_sum += loss.item()
    sched.step()

    model.eval()
    correct = total = 0
    with torch.no_grad():
        for x, y in val_dl:
            x, y = x.to(device), y.to(device)
            correct += (model(x).argmax(1) == y).sum().item()
            total   += y.size(0)
    acc = correct / total
    print(f"epoch {epoch:02d}  loss {loss_sum/len(train_dl):.3f}  val_acc {acc:.4f}")

    if acc > best_acc:
        best_acc = acc
        torch.save(model.state_dict(), "/kaggle/working/alexnet_imagenette.pt")

print("best val_acc:", best_acc)

epoch 01  loss 2.301  val_acc 0.1306
epoch 02  loss 2.269  val_acc 0.2162
epoch 03  loss 2.083  val_acc 0.2956
epoch 04  loss 1.881  val_acc 0.3537
epoch 05  loss 1.768  val_acc 0.4530
epoch 06  loss 1.620  val_acc 0.5118
epoch 07  loss 1.480  val_acc 0.5615
epoch 08  loss 1.360  val_acc 0.5378
epoch 09  loss 1.239  val_acc 0.6196
epoch 10  loss 1.134  val_acc 0.6662
epoch 11  loss 1.062  val_acc 0.6150
epoch 12  loss 0.991  val_acc 0.7021
epoch 13  loss 0.929  val_acc 0.7242
epoch 14  loss 0.919  val_acc 0.7135
epoch 15  loss 0.851  val_acc 0.7471
epoch 16  loss 0.677  val_acc 0.7624
epoch 17  loss 0.640  val_acc 0.7693
epoch 18  loss 0.630  val_acc 0.7662
epoch 19  loss 0.618  val_acc 0.7716
epoch 20  loss 0.613  val_acc 0.7777
epoch 21  loss 0.582  val_acc 0.7739
epoch 22  loss 0.576  val_acc 0.7792
epoch 23  loss 0.576  val_acc 0.7823
epoch 24  loss 0.565  val_acc 0.7876
epoch 25  loss 0.559  val_acc 0.7884
epoch 26  loss 0.553  val_acc 0.7899
epoch 27  loss 0.547  val_acc 0.7869
e

In [7]:
model.load_state_dict(torch.load("/kaggle/working/alexnet_imagenette.pt"))
model.eval()
correct = total = 0
with torch.no_grad():
    for x, y in test_dl:
        x, y = x.to(device), y.to(device)
        correct += (model(x).argmax(1) == y).sum().item()
        total   += y.size(0)
print(f"test_acc: {correct/total:.4f}")

test_acc: 0.7947
